In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Libraries

In [22]:
import numpy as np
import pandas as pd
import warnings

from sklearn.base import clone
from sklearn.linear_model import (
    LinearRegression, Ridge, RidgeCV, Lasso, LassoCV,
    ElasticNet, ElasticNetCV, BayesianRidge
)
from sklearn.cross_decomposition import PLSRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, BaggingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.dummy import DummyRegressor
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Load datasets

In [18]:
barley = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_06_NIRs_calibration/barley.csv')
legumes = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_06_NIRs_calibration/legumes.csv')

In [21]:
legumes.head(2)

,position,ciat_lab_id,id_lmf,tax_name,dm,ash,protein,adf,ndf,net_gas_8h_ml,...,2495,2495.5,2496,2496.5,2497,2497.5,2498,2498.5,2499,2499.5
0,1,F243416,ABC-705-1,Indigofera suffruticosa,91.97,12.68,33.01,28.05,45.22,29.35,...,0.592290,0.592270,0.592244,0.592214,0.592179,0.592138,0.592091,0.592035,0.591970,0.591889
1,2,F243417,ABC-707-1,Alysicarpus ovalifolius,90.09,12.07,25.40,25.44,54.11,30.58,...,0.493827,0.493835,0.493839,0.493836,0.493827,0.493809,0.493783,0.493746,0.493696,0.493628


# Barley

In [19]:
barley.head(2)

,sample,dm,ash,om,ndf,adf,protein,tddm,gas_g_dm_incubach4_g_dm_incubatch4_g_digested,ch4_g_dm_m_incubatch4_digested,...,2480,2482,2484,2486,2488,2490,2492,2494,2496,2498
0,403,94.089996,14.32,85.68,48.430000,24.930000,17.110001,84.410004,212.190002,30.990000,...,0.565927,0.567928,0.569845,0.571519,0.572804,0.573721,0.574330,0.574704,0.574816,0.574592
1,404,94.150002,12.50,87.50,55.759998,24.790001,15.900000,78.760002,230.440002,32.729999,...,0.547116,0.549239,0.551228,0.552993,0.554383,0.555349,0.555955,0.556324,0.556430,0.556195


## Ash

In [23]:
# Global seed
np.random.seed(42)

# =========================
# 2. Load dataset
# =========================
df = barley

# =========================
# 3. Prepare data
# =========================
def is_numeric_col(col):
    try:
        float(col)  # acepta enteros y decimales
        return True
    except ValueError:
        return False

identifier_cols = ["sample"]

X_cols = [
    col for col in df.columns
    if is_numeric_col(col)
    and col not in identifier_cols
    and col != "ash"
]

# Ordenar temporalmente
X_cols = sorted(X_cols, key=lambda c: float(c))

y = df["ash"].to_numpy()
X = df[X_cols].to_numpy()

# =========================
# 4. Train-test split
# =========================
n = len(X)
split = int(0.8 * n)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# =========================
# 5. Standardization
# =========================
mu = np.mean(X_train, axis=0)
sigma = np.std(X_train, axis=0)

# evitar división por cero
sigma[sigma == 0] = 1

X_train_s = (X_train - mu) / sigma
X_test_s = (X_test - mu) / sigma

# =========================
# 6. Models
# =========================
def set_random_state(model, seed=42):

    if hasattr(model, "random_state"):
        try:
            model.set_params(random_state=seed)
        except:
            pass
    return model

models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "RidgeCV": RidgeCV(),
    "Lasso": Lasso(),
    "LassoCV": LassoCV(),
    "ElasticNet": ElasticNet(),
    "ElasticNetCV": ElasticNetCV(),
    "BayesianRidge": BayesianRidge(),
    "PLS": PLSRegression(),
    "DecisionTree": DecisionTreeRegressor(),
    "RandomForest": RandomForestRegressor(),
    "ExtraTrees": ExtraTreesRegressor(),
    "Bagging": BaggingRegressor(),
    "SVR": SVR(),
    "KNN": KNeighborsRegressor(),
    "Dummy": DummyRegressor()
}

# Aplicar random_state automáticamente
models = {name: set_random_state(model) for name, model in models.items()}

# =========================
# 7. Metrics
# =========================
def r2_score_manual(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot

def mse_manual(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def pearson_r(y_true, y_pred):
    y_mean = np.mean(y_true)
    yhat_mean = np.mean(y_pred)

    num = np.sum((y_true - y_mean) * (y_pred - yhat_mean))
    den = np.sqrt(
        np.sum((y_true - y_mean) ** 2) *
        np.sum((y_pred - yhat_mean) ** 2)
    )

    if den == 0:
        return 0

    return num / den

# =========================
# 8. Train and evaluate
# =========================
def evaluate_models(X_tr, X_te, y_tr, y_te, models):
    results = []

    for name, model in models.items():
        try:
            fitted_model = clone(model)
            fitted_model.fit(X_tr, y_tr)

            y_pred = fitted_model.predict(X_te)

            if isinstance(y_pred, np.ndarray) and y_pred.ndim > 1:
                y_pred = y_pred.ravel()

            r = pearson_r(y_te, y_pred)
            r2 = r2_score_manual(y_te, y_pred)
            err = mse_manual(y_te, y_pred)

            results.append([name, r, r2, err])
        except:
            # mantener robustez sin romper ejecución
            pass

    return results

results_raw = evaluate_models(X_train, X_test, y_train, y_test, models)
results_std = evaluate_models(X_train_s, X_test_s, y_train, y_test, models)

# =========================
# 9. Tables
# =========================
df_raw = pd.DataFrame(results_raw, columns=["Model", "r", "R2", "MSE"])
df_std = pd.DataFrame(results_std, columns=["Model", "r", "R2", "MSE"])

df_raw = df_raw.sort_values(by="R2", ascending=False).reset_index(drop=True)
df_std = df_std.sort_values(by="R2", ascending=False).reset_index(drop=True)

print("Table 1: Raw Data")
print(df_raw)

print("\nTable 2: Standardized Data")
print(df_std)

# =========================
# 10. Standard deviation of ALL R²
# =========================
r2_all = pd.concat([df_raw["R2"], df_std["R2"]], ignore_index=True).to_numpy()

mean_r2 = np.mean(r2_all)

sigma_r2 = np.sqrt(
    np.sum((r2_all - mean_r2) ** 2) / (len(r2_all) - 1)
)


Table 1: Raw Data
               Model             r        R2       MSE
0       ElasticNetCV  6.998496e-01  0.272412  4.331353
1            LassoCV  5.985320e-01  0.225300  4.611808
2            RidgeCV  6.405644e-01  0.196405  4.783823
3   LinearRegression  5.117101e-01  0.180065  4.881096
4      BayesianRidge  5.875619e-01  0.145935  5.084274
5         ExtraTrees  5.431217e-01  0.142032  5.107506
6              Ridge  5.669130e-01  0.113765  5.275783
7                PLS  5.556284e-01  0.106736  5.317625
8       RandomForest  5.882019e-01  0.106247  5.320538
9            Bagging  5.106996e-01  0.094774  5.388834
10               KNN  5.037676e-01  0.042414  5.700534
11      DecisionTree  4.677853e-01  0.006524  5.914191
12        ElasticNet -1.820125e-16 -0.241944  7.393328
13             Lasso -1.820125e-16 -0.241944  7.393328
14             Dummy -1.820125e-16 -0.241944  7.393328
15               SVR  4.541033e-01 -0.273064  7.578587

Table 2: Standardized Data
               Mode

In [25]:
# =========================================================
# GLOBAL SEED
# =========================================================
np.random.seed(42)


# =========================================================
# AUXILIARY FUNCTIONS
# =========================================================
def is_numeric_col(col):
    """
    Detects columns whose names are numeric.
    """
    try:
        float(col)
        return True
    except:
        return False


def set_random_state(model, seed=42):
    """
    Automatically assigns random_state when supported.
    """
    if hasattr(model, "random_state"):
        try:
            model.set_params(random_state=seed)
        except:
            pass

    return model


def r2_score_manual(y_true, y_pred):

    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)

    return 1 - ss_res / ss_tot


def mse_manual(y_true, y_pred):

    return np.mean((y_true - y_pred) ** 2)


def pearson_r(y_true, y_pred):

    y_mean = np.mean(y_true)
    yhat_mean = np.mean(y_pred)

    numerator = np.sum((y_true - y_mean) * (y_pred - yhat_mean))

    denominator = np.sqrt(
        np.sum((y_true - y_mean) ** 2) *
        np.sum((y_pred - yhat_mean) ** 2)
    )

    if denominator == 0:
        return 0

    return numerator / denominator


def evaluate_models(X_tr, X_te, y_tr, y_te, models):

    results = []

    for name, model in models.items():

        try:
            fitted_model = clone(model)

            fitted_model.fit(X_tr, y_tr)

            y_pred = fitted_model.predict(X_te)

            if isinstance(y_pred, np.ndarray) and y_pred.ndim > 1:
                y_pred = y_pred.ravel()

            r = pearson_r(y_te, y_pred)
            r2 = r2_score_manual(y_te, y_pred)
            mse = mse_manual(y_te, y_pred)

            results.append([name, r, r2, mse])

        except:
            pass

    return results


# =========================================================
# MAIN FUNCTION
# =========================================================
def evaluate_trait_models(
    df,
    target_trait,
    identifier_cols=["sample"],
    test_size=0.2,
    seed=42
):
    """
    Evaluates multiple regression models for a selected trait.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataset.

    target_trait : str
        Trait/variable to predict.

    identifier_cols : list
        Columns excluded from predictors.

    test_size : float
        Proportion of test data.

    seed : int
        Random seed.

    Returns
    -------
    dict containing:
        - raw_results
        - standardized_results
        - sigma_r2
        - mean_r2
    """

    np.random.seed(seed)

    # =====================================================
    # SELECT PREDICTOR VARIABLES
    # =====================================================
    X_cols = [
        col for col in df.columns
        if is_numeric_col(col)
        and col not in identifier_cols
        and col != target_trait
    ]

    X_cols = sorted(X_cols, key=lambda c: float(c))

    # =====================================================
    # MATRICES
    # =====================================================
    y = df[target_trait].to_numpy()
    X = df[X_cols].to_numpy()

    # =====================================================
    # TRAIN / TEST SPLIT
    # =====================================================
    n = len(X)

    split = int((1 - test_size) * n)

    X_train = X[:split]
    X_test = X[split:]

    y_train = y[:split]
    y_test = y[split:]

    # =====================================================
    # STANDARDIZATION
    # =====================================================
    mu = np.mean(X_train, axis=0)

    sigma = np.std(X_train, axis=0)

    sigma[sigma == 0] = 1

    X_train_s = (X_train - mu) / sigma
    X_test_s = (X_test - mu) / sigma

    # =====================================================
    # MODELS
    # =====================================================
    models = {
        "LinearRegression": LinearRegression(),
        "Ridge": Ridge(),
        "RidgeCV": RidgeCV(),
        "Lasso": Lasso(),
        "LassoCV": LassoCV(),
        "ElasticNet": ElasticNet(),
        "ElasticNetCV": ElasticNetCV(),
        "BayesianRidge": BayesianRidge(),
        "PLS": PLSRegression(),
        "DecisionTree": DecisionTreeRegressor(),
        "RandomForest": RandomForestRegressor(),
        "ExtraTrees": ExtraTreesRegressor(),
        "Bagging": BaggingRegressor(),
        "SVR": SVR(),
        "KNN": KNeighborsRegressor(),
        "Dummy": DummyRegressor()
    }

    models = {
        name: set_random_state(model, seed)
        for name, model in models.items()
    }

    # =====================================================
    # EVALUATION
    # =====================================================
    results_raw = evaluate_models(
        X_train,
        X_test,
        y_train,
        y_test,
        models
    )

    results_std = evaluate_models(
        X_train_s,
        X_test_s,
        y_train,
        y_test,
        models
    )

    # =====================================================
    # TABLES
    # =====================================================
    df_raw = pd.DataFrame(
        results_raw,
        columns=["Model", "r", "R2", "MSE"]
    )

    df_std = pd.DataFrame(
        results_std,
        columns=["Model", "r", "R2", "MSE"]
    )

    df_raw = (
        df_raw
        .sort_values(by="R2", ascending=False)
        .reset_index(drop=True)
    )

    df_std = (
        df_std
        .sort_values(by="R2", ascending=False)
        .reset_index(drop=True)
    )

    # =====================================================
    # STANDARD DEVIATION OF ALL R²
    # =====================================================
    r2_all = pd.concat(
        [df_raw["R2"], df_std["R2"]],
        ignore_index=True
    ).to_numpy()

    mean_r2 = np.mean(r2_all)

    sigma_r2 = np.sqrt(
        np.sum((r2_all - mean_r2) ** 2) /
        (len(r2_all) - 1)
    )

    # =====================================================
    # PRINT RESULTS
    # =====================================================
    print(f"\nTarget trait: {target_trait}")

    print("\nTable 1: Raw Data")
    print(df_raw)

    print("\nTable 2: Standardized Data")
    print(df_std)

    print("\nMean R²:", mean_r2)
    print("SD of R²:", sigma_r2)

    # =====================================================
    # RETURN RESULTS
    # =====================================================
    return {
        "raw_results": df_raw,
        "standardized_results": df_std,
        "mean_r2": mean_r2,
        "sigma_r2": sigma_r2
    }


# =========================================================
# EXAMPLE
# =========================================================

results = evaluate_trait_models(
    df=barley,
    target_trait="ash"
)


Target trait: ash

Table 1: Raw Data
               Model             r        R2       MSE
0       ElasticNetCV  6.998496e-01  0.272412  4.331353
1            LassoCV  5.985320e-01  0.225300  4.611808
2            RidgeCV  6.405644e-01  0.196405  4.783823
3   LinearRegression  5.117101e-01  0.180065  4.881096
4      BayesianRidge  5.875619e-01  0.145935  5.084274
5         ExtraTrees  5.431217e-01  0.142032  5.107506
6              Ridge  5.669130e-01  0.113765  5.275783
7                PLS  5.556284e-01  0.106736  5.317625
8       RandomForest  5.882019e-01  0.106247  5.320538
9            Bagging  5.106996e-01  0.094774  5.388834
10               KNN  5.037676e-01  0.042414  5.700534
11      DecisionTree  4.677853e-01  0.006524  5.914191
12        ElasticNet -1.820125e-16 -0.241944  7.393328
13             Lasso -1.820125e-16 -0.241944  7.393328
14             Dummy -1.820125e-16 -0.241944  7.393328
15               SVR  4.541033e-01 -0.273064  7.578587

Table 2: Standardized Data

In [ ]:
barley_ash = evaluate_trait_models(df=barley, target_trait="ash")
barley_ash

# dm

In [24]:
barley_dm = evaluate_trait_models(df=barley, target_trait="dm")
barley_dm

Index(['sample', 'dm', 'ash', 'om', 'ndf', 'adf', 'protein', 'tddm',
       'gas_g_dm_incubach4_g_dm_incubatch4_g_digested',
       'ch4_g_dm_m_incubatch4_digested',
       ...
       '2480', '2482', '2484', '2486', '2488', '2490', '2492', '2494', '2496',
       '2498'],
      dtype='object', length=1064)

# om

In [ ]:
# om

# Manual Model Evaluation (Code 1, all dataset) legumes_short

In [ ]:
legumes_short = legumes.iloc[:50]
legumes_half = legumes.iloc[:160]
legumes_large = legumes.copy()

In [ ]:
# =========================
# 1. Import libraries
# =========================
import numpy as np
import pandas as pd
import warnings

from sklearn.base import clone
from sklearn.linear_model import (
    LinearRegression, Ridge, RidgeCV, Lasso, LassoCV,
    ElasticNet, ElasticNetCV, BayesianRidge
)
from sklearn.cross_decomposition import PLSRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, BaggingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.dummy import DummyRegressor
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Global seed
np.random.seed(42)

# =========================
# 2. Load dataset
# =========================
df = legumes_short

# =========================
# 3. Prepare data
# =========================
def is_numeric_col(col):
    try:
        float(col)  # acepta enteros y decimales
        return True
    except ValueError:
        return False

identifier_cols = ["sample"]

X_cols = [
    col for col in df.columns
    if is_numeric_col(col)
    and col not in identifier_cols
    and col != "ash"
]

# Ordenar temporalmente
X_cols = sorted(X_cols, key=lambda c: float(c))

y = df["ash"].to_numpy()
X = df[X_cols].to_numpy()

# =========================
# 4. Train-test split
# =========================
n = len(X)
split = int(0.8 * n)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# =========================
# 5. Standardization
# =========================
mu = np.mean(X_train, axis=0)
sigma = np.std(X_train, axis=0)

# evitar división por cero
sigma[sigma == 0] = 1

X_train_s = (X_train - mu) / sigma
X_test_s = (X_test - mu) / sigma

# =========================
# 6. Models
# =========================
def set_random_state(model, seed=42):

    if hasattr(model, "random_state"):
        try:
            model.set_params(random_state=seed)
        except:
            pass
    return model

models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "RidgeCV": RidgeCV(),
    "Lasso": Lasso(),
    "LassoCV": LassoCV(),
    "ElasticNet": ElasticNet(),
    "ElasticNetCV": ElasticNetCV(),
    "BayesianRidge": BayesianRidge(),
    "PLS": PLSRegression(),
    "DecisionTree": DecisionTreeRegressor(),
    "RandomForest": RandomForestRegressor(),
    "ExtraTrees": ExtraTreesRegressor(),
    "Bagging": BaggingRegressor(),
    "SVR": SVR(),
    "KNN": KNeighborsRegressor(),
    "Dummy": DummyRegressor()
}

# Aplicar random_state automáticamente
models = {name: set_random_state(model) for name, model in models.items()}

# =========================
# 7. Metrics
# =========================
def r2_score_manual(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot

def mse_manual(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def pearson_r(y_true, y_pred):
    y_mean = np.mean(y_true)
    yhat_mean = np.mean(y_pred)

    num = np.sum((y_true - y_mean) * (y_pred - yhat_mean))
    den = np.sqrt(
        np.sum((y_true - y_mean) ** 2) *
        np.sum((y_pred - yhat_mean) ** 2)
    )

    if den == 0:
        return 0

    return num / den

# =========================
# 8. Train and evaluate
# =========================
def evaluate_models(X_tr, X_te, y_tr, y_te, models):
    results = []

    for name, model in models.items():
        try:
            fitted_model = clone(model)
            fitted_model.fit(X_tr, y_tr)

            y_pred = fitted_model.predict(X_te)

            if isinstance(y_pred, np.ndarray) and y_pred.ndim > 1:
                y_pred = y_pred.ravel()

            r = pearson_r(y_te, y_pred)
            r2 = r2_score_manual(y_te, y_pred)
            err = mse_manual(y_te, y_pred)

            results.append([name, r, r2, err])
        except:
            # mantener robustez sin romper ejecución
            pass

    return results

results_raw = evaluate_models(X_train, X_test, y_train, y_test, models)
results_std = evaluate_models(X_train_s, X_test_s, y_train, y_test, models)

# =========================
# 9. Tables
# =========================
df_raw = pd.DataFrame(results_raw, columns=["Model", "r", "R2", "MSE"])
df_std = pd.DataFrame(results_std, columns=["Model", "r", "R2", "MSE"])

df_raw = df_raw.sort_values(by="R2", ascending=False).reset_index(drop=True)
df_std = df_std.sort_values(by="R2", ascending=False).reset_index(drop=True)

print("Table 1: Raw Data")
print(df_raw)

print("\nTable 2: Standardized Data")
print(df_std)

# =========================
# 10. Standard deviation of ALL R²
# =========================
r2_all = pd.concat([df_raw["R2"], df_std["R2"]], ignore_index=True).to_numpy()

mean_r2 = np.mean(r2_all)

sigma_r2 = np.sqrt(
    np.sum((r2_all - mean_r2) ** 2) / (len(r2_all) - 1)
)


Table 1: Raw Data
               Model         r        R2        MSE
0            LassoCV  0.769314  0.441069   3.193555
1   LinearRegression  0.684777  0.172192   4.729828
2       ElasticNetCV  0.694357  0.076824   5.274735
3                SVR  0.208580 -0.444068   8.250943
4            RidgeCV  0.572816 -0.563844   8.935309
5      BayesianRidge  0.529740 -0.729816   9.883618
6         ElasticNet  0.000000 -0.799830  10.283656
7              Lasso  0.000000 -0.799830  10.283656
8              Dummy  0.000000 -0.799830  10.283656
9                KNN  0.355604 -0.895845  10.832251
10           Bagging  0.370459 -1.416998  13.809956
11             Ridge  0.359103 -1.458373  14.046361
12      RandomForest  0.346934 -1.626713  15.008199
13        ExtraTrees  0.320330 -2.441931  19.666098
14               PLS  0.227547 -2.458905  19.763081
15      DecisionTree  0.346544 -2.486311  19.919670

Table 2: Standardized Data
               Model         r        R2        MSE
0              Rid

In [ ]:
# =========================
# 1. Import libraries
# =========================
import numpy as np
import pandas as pd
import warnings

from sklearn.base import clone
from sklearn.linear_model import (
    LinearRegression, Ridge, RidgeCV, Lasso, LassoCV,
    ElasticNet, ElasticNetCV, BayesianRidge
)
from sklearn.cross_decomposition import PLSRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, BaggingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.dummy import DummyRegressor
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Global seed
np.random.seed(42)

# =========================
# 2. Load dataset
# =========================
df = legumes_half

# =========================
# 3. Prepare data
# =========================
def is_numeric_col(col):
    try:
        float(col)  # acepta enteros y decimales
        return True
    except ValueError:
        return False

identifier_cols = ["sample"]

X_cols = [
    col for col in df.columns
    if is_numeric_col(col)
    and col not in identifier_cols
    and col != "ash"
]

# Ordenar temporalmente
X_cols = sorted(X_cols, key=lambda c: float(c))

y = df["ash"].to_numpy()
X = df[X_cols].to_numpy()

# =========================
# 4. Train-test split
# =========================
n = len(X)
split = int(0.8 * n)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# =========================
# 5. Standardization
# =========================
mu = np.mean(X_train, axis=0)
sigma = np.std(X_train, axis=0)

# evitar división por cero
sigma[sigma == 0] = 1

X_train_s = (X_train - mu) / sigma
X_test_s = (X_test - mu) / sigma

# =========================
# 6. Models
# =========================
def set_random_state(model, seed=42):

    if hasattr(model, "random_state"):
        try:
            model.set_params(random_state=seed)
        except:
            pass
    return model

models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "RidgeCV": RidgeCV(),
    "Lasso": Lasso(),
    "LassoCV": LassoCV(),
    "ElasticNet": ElasticNet(),
    "ElasticNetCV": ElasticNetCV(),
    "BayesianRidge": BayesianRidge(),
    "PLS": PLSRegression(),
    "DecisionTree": DecisionTreeRegressor(),
    "RandomForest": RandomForestRegressor(),
    "ExtraTrees": ExtraTreesRegressor(),
    "Bagging": BaggingRegressor(),
    "SVR": SVR(),
    "KNN": KNeighborsRegressor(),
    "Dummy": DummyRegressor()
}

# Aplicar random_state automáticamente
models = {name: set_random_state(model) for name, model in models.items()}

# =========================
# 7. Metrics
# =========================
def r2_score_manual(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot

def mse_manual(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def pearson_r(y_true, y_pred):
    y_mean = np.mean(y_true)
    yhat_mean = np.mean(y_pred)

    num = np.sum((y_true - y_mean) * (y_pred - yhat_mean))
    den = np.sqrt(
        np.sum((y_true - y_mean) ** 2) *
        np.sum((y_pred - yhat_mean) ** 2)
    )

    if den == 0:
        return 0

    return num / den

# =========================
# 8. Train and evaluate
# =========================
def evaluate_models(X_tr, X_te, y_tr, y_te, models):
    results = []

    for name, model in models.items():
        try:
            fitted_model = clone(model)
            fitted_model.fit(X_tr, y_tr)

            y_pred = fitted_model.predict(X_te)

            if isinstance(y_pred, np.ndarray) and y_pred.ndim > 1:
                y_pred = y_pred.ravel()

            r = pearson_r(y_te, y_pred)
            r2 = r2_score_manual(y_te, y_pred)
            err = mse_manual(y_te, y_pred)

            results.append([name, r, r2, err])
        except:
            # mantener robustez sin romper ejecución
            pass

    return results

results_raw = evaluate_models(X_train, X_test, y_train, y_test, models)
results_std = evaluate_models(X_train_s, X_test_s, y_train, y_test, models)

# =========================
# 9. Tables
# =========================
df_raw = pd.DataFrame(results_raw, columns=["Model", "r", "R2", "MSE"])
df_std = pd.DataFrame(results_std, columns=["Model", "r", "R2", "MSE"])

df_raw = df_raw.sort_values(by="R2", ascending=False).reset_index(drop=True)
df_std = df_std.sort_values(by="R2", ascending=False).reset_index(drop=True)

print("Table 1: Raw Data")
print(df_raw)

print("\nTable 2: Standardized Data")
print(df_std)

# =========================
# 10. Standard deviation of ALL R²
# =========================
r2_all = pd.concat([df_raw["R2"], df_std["R2"]], ignore_index=True).to_numpy()

mean_r2 = np.mean(r2_all)

sigma_r2 = np.sqrt(
    np.sum((r2_all - mean_r2) ** 2) / (len(r2_all) - 1)
)


Table 1: Raw Data
               Model         r        R2        MSE
0            LassoCV  0.580061 -0.009158   4.310939
1                SVR  0.593272 -0.034519   4.419279
2         ElasticNet  0.000000 -0.126705   4.813077
3              Lasso  0.000000 -0.126705   4.813077
4              Dummy  0.000000 -0.126705   4.813077
5      BayesianRidge  0.638982 -0.161927   4.963539
6       ElasticNetCV  0.628464 -0.421520   6.072476
7                KNN  0.369254 -0.436001   6.134334
8         ExtraTrees  0.465656 -0.665540   7.114883
9       RandomForest  0.523717 -0.786930   7.633438
10  LinearRegression  0.443352 -0.820678   7.777605
11           RidgeCV  0.655670 -1.546233  10.877044
12      DecisionTree  0.501193 -1.623684  11.207900
13           Bagging  0.456584 -2.168417  13.534899
14             Ridge  0.614858 -3.008749  17.124646
15               PLS  0.569102 -4.212567  22.267135

Table 2: Standardized Data
               Model         r        R2        MSE
0       ElasticNet

In [ ]:
# =========================
# 1. Import libraries
# =========================
import numpy as np
import pandas as pd
import warnings

from sklearn.base import clone
from sklearn.linear_model import (
    LinearRegression, Ridge, RidgeCV, Lasso, LassoCV,
    ElasticNet, ElasticNetCV, BayesianRidge
)
from sklearn.cross_decomposition import PLSRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, BaggingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.dummy import DummyRegressor
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Global seed
np.random.seed(42)

# =========================
# 2. Load dataset
# =========================
df = legumes_large

# =========================
# 3. Prepare data
# =========================
def is_numeric_col(col):
    try:
        float(col)  # acepta enteros y decimales
        return True
    except ValueError:
        return False

identifier_cols = ["sample"]

X_cols = [
    col for col in df.columns
    if is_numeric_col(col)
    and col not in identifier_cols
    and col != "ash"
]

# Ordenar temporalmente
X_cols = sorted(X_cols, key=lambda c: float(c))

y = df["ash"].to_numpy()
X = df[X_cols].to_numpy()

# =========================
# 4. Train-test split
# =========================
n = len(X)
split = int(0.8 * n)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# =========================
# 5. Standardization
# =========================
mu = np.mean(X_train, axis=0)
sigma = np.std(X_train, axis=0)

# evitar división por cero
sigma[sigma == 0] = 1

X_train_s = (X_train - mu) / sigma
X_test_s = (X_test - mu) / sigma

# =========================
# 6. Models
# =========================
def set_random_state(model, seed=42):

    if hasattr(model, "random_state"):
        try:
            model.set_params(random_state=seed)
        except:
            pass
    return model

models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "RidgeCV": RidgeCV(),
    "Lasso": Lasso(),
    "LassoCV": LassoCV(),
    "ElasticNet": ElasticNet(),
    "ElasticNetCV": ElasticNetCV(),
    "BayesianRidge": BayesianRidge(),
    "PLS": PLSRegression(),
    "DecisionTree": DecisionTreeRegressor(),
    "RandomForest": RandomForestRegressor(),
    "ExtraTrees": ExtraTreesRegressor(),
    "Bagging": BaggingRegressor(),
    "SVR": SVR(),
    "KNN": KNeighborsRegressor(),
    "Dummy": DummyRegressor()
}

# Aplicar random_state automáticamente
models = {name: set_random_state(model) for name, model in models.items()}

# =========================
# 7. Metrics
# =========================
def r2_score_manual(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot

def mse_manual(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def pearson_r(y_true, y_pred):
    y_mean = np.mean(y_true)
    yhat_mean = np.mean(y_pred)

    num = np.sum((y_true - y_mean) * (y_pred - yhat_mean))
    den = np.sqrt(
        np.sum((y_true - y_mean) ** 2) *
        np.sum((y_pred - yhat_mean) ** 2)
    )

    if den == 0:
        return 0

    return num / den

# =========================
# 8. Train and evaluate
# =========================
def evaluate_models(X_tr, X_te, y_tr, y_te, models):
    results = []

    for name, model in models.items():
        try:
            fitted_model = clone(model)
            fitted_model.fit(X_tr, y_tr)

            y_pred = fitted_model.predict(X_te)

            if isinstance(y_pred, np.ndarray) and y_pred.ndim > 1:
                y_pred = y_pred.ravel()

            r = pearson_r(y_te, y_pred)
            r2 = r2_score_manual(y_te, y_pred)
            err = mse_manual(y_te, y_pred)

            results.append([name, r, r2, err])
        except:
            # mantener robustez sin romper ejecución
            pass

    return results

results_raw = evaluate_models(X_train, X_test, y_train, y_test, models)
results_std = evaluate_models(X_train_s, X_test_s, y_train, y_test, models)

# =========================
# 9. Tables
# =========================
df_raw = pd.DataFrame(results_raw, columns=["Model", "r", "R2", "MSE"])
df_std = pd.DataFrame(results_std, columns=["Model", "r", "R2", "MSE"])

df_raw = df_raw.sort_values(by="R2", ascending=False).reset_index(drop=True)
df_std = df_std.sort_values(by="R2", ascending=False).reset_index(drop=True)

print("Table 1: Raw Data")
print(df_raw)

print("\nTable 2: Standardized Data")
print(df_std)

# =========================
# 10. Standard deviation of ALL R²
# =========================
r2_all = pd.concat([df_raw["R2"], df_std["R2"]], ignore_index=True).to_numpy()

mean_r2 = np.mean(r2_all)

sigma_r2 = np.sqrt(
    np.sum((r2_all - mean_r2) ** 2) / (len(r2_all) - 1)
)


Table 1: Raw Data
               Model         r        R2        MSE
0      BayesianRidge  0.722224 -0.123348   5.879413
1       ElasticNetCV  0.605069 -0.293896   6.772035
2            RidgeCV  0.682833 -0.431951   7.494589
3                SVR  0.361290 -0.694218   8.867249
4                PLS  0.372177 -0.969931  10.310289
5              Ridge  0.591833 -1.059183  10.777414
6         ElasticNet  0.000000 -1.691410  14.086384
7              Lasso  0.000000 -1.691410  14.086384
8              Dummy  0.000000 -1.691410  14.086384
9            LassoCV  0.364936 -1.851623  14.924914
10      RandomForest  0.307354 -2.491525  18.274054
11        ExtraTrees  0.278068 -2.506551  18.352695
12           Bagging  0.319919 -2.658728  19.149166
13               KNN  0.176405 -2.678325  19.251732
14  LinearRegression  0.530769 -4.308247  27.782467
15      DecisionTree  0.086907 -5.483741  33.934803

Table 2: Standardized Data
               Model         r        R2        MSE
0              Rid

# Manual Model evaluation (Code 2 All dataset)

In [ ]:
# Import basic libraries
import numpy as np
import pandas as pd
import time

# Import only necessary sklearn models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.dummy import DummyRegressor

# NEW: PLS
from sklearn.cross_decomposition import PLSRegression

# Set random seed for reproducibility
np.random.seed(42)

# =========================
# 1. Load dataset
# =========================
df = llegumes_short

# =========================
# 2. Separate X and Y
# =========================
X_cols = [col for col in df.columns if col.isdigit()]

y = df["ash"].values
X = df[X_cols].values

# =========================
# 3. Train-test split (80/20, ordered)
# =========================
n = len(X)
split = int(0.8 * n)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# =========================
# 4. Define models (same structure + PLS)
# =========================
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "ElasticNet": ElasticNet(),
    "BayesianRidge": BayesianRidge(),
    "PLS": PLSRegression(n_components=10),
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(random_state=42),
    "ExtraTrees": ExtraTreesRegressor(random_state=42),
    "Bagging": BaggingRegressor(random_state=42),
    "SVR": SVR(),
    "KNN": KNeighborsRegressor(),
    "Dummy": DummyRegressor()
}

# =========================
# 5. Metric functions (manual)
# =========================
def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    return 1 - ss_res / ss_tot

def adjusted_r2(r2, m, p):
    return 1 - (1 - r2) * (m - 1) / (m - p - 1)

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

# =========================
# 6. Train and evaluate models
# =========================
results = []
r2_list = []

m = len(y_test)
p = X.shape[1]

for name, model in models.items():
    try:
        start = time.time()

        model.fit(X_train, y_train)

        # IMPORTANT: ensure 1D output (PLS fix)
        y_pred = model.predict(X_test)
        y_pred = np.ravel(y_pred)

        end = time.time()

        r2 = r2_score(y_test, y_pred)
        adj_r2 = adjusted_r2(r2, m, p)
        error = rmse(y_test, y_pred)
        t = end - start

        results.append([name, adj_r2, r2, error, t])
        r2_list.append(r2)

    except:
        continue

# =========================
# 7. Create results table
# =========================
results_df = pd.DataFrame(results, columns=[
    "Model", "Adjusted R-Squared", "R-Squared", "RMSE", "Time Taken"
])

results_df = results_df.sort_values(by="R-Squared", ascending=False)

print(results_df)

# =========================
# 8. Compute std deviation of R²
# =========================
r2_array = np.array(r2_list)
r2_mean = np.mean(r2_array)

sigma_r2 = np.sqrt(np.sum((r2_array - r2_mean)**2) / (len(r2_array) - 1))

# =========================
# 9. Final answer
# =========================
print("\nFinal answer:", format(sigma_r2, ".3g"))

# Model evaluations with Lazy Predict

In [ ]:
!pip install lazypredict

In [ ]:
from lazypredict.Supervised import LazyRegressor

## Raw data

In [ ]:
df = legumes.copy()

In [ ]:
# Select numeric column names (NIRS predictors)
X_cols = [col for col in df.columns if col.isdigit()]

# Target variable
y = df["ash"].values

# Predictor matrix
X = df[X_cols].values

In [ ]:

X_train, X_test, y_train, y_test = train_test_split (X, y, test_size = 0.20, random_state = 1)

reg = LazyRegressor(verbose=0, ignore_warnings=False, custom_metric=None)
models, predictions = reg.fit(X_train, X_test, y_train, y_test)

models

## Standardized data

In [ ]:
df = legumes.copy()

In [ ]:
# Select numeric column names (NIRS predictors)
X_cols = [col for col in df.columns if col.isdigit()]

# Target variable
y = df["ash"].values

# Predictor matrix
X = df[X_cols].values

# Data split
X_train, X_test, y_train, y_test = train_test_split (X, y, test_size = 0.20, random_state = 1)

In [ ]:

# Standardization
scaler = StandardScaler()

# Adjust and transform training data
X_train_scaled = scaler.fit_transform(X_train)

# transform testing data
X_test_scaled = scaler.transform(X_test)


In [ ]:

reg = LazyRegressor(verbose=0, ignore_warnings=False, custom_metric=None)
models, predictions = reg.fit(X_train_scaled, X_test_scaled, y_train, y_test)

models